**[🏠 Course Home](../README.md)** | **📖 [Conceptual Track](../conceptual/13_hierarchical_models_and_partial_pooling.ipynb)** | **🤖 [AI Track](13_hierarchical_models_and_partial_pooling.ipynb)** | **🐍 [Python Track](../python/13_hierarchical_models_and_partial_pooling.ipynb)** | **🔵 [R Track](../r/13_hierarchical_models_and_partial_pooling.ipynb)** | **⬅️ [Prev: Ch 12: Bayesian Regression](12_bayesian_regression_and_glms.ipynb)** | **➡️ [Next: Ch 14: Marginal Likelihood](14_marginal_likelihood_and_model_comparison.ipynb)**

*🧭 Part III: Applied Modeling & Hierarchy — Chapter 13 of 22*

---

# Chapter 13: Hierarchical Models & Partial Pooling
### *Prompting for Multi-Group Data, Stein's Paradox, and Neal's Funnel*

---

> **🧭 Course Context & Why This Module Matters in the AI Era:**
>
> In real-world software and hardware systems, engineers rarely evaluate
> isolated components in a vacuum. Instead, systems are partitioned into
> multi-group hierarchies:
> * 12 regional cloud clusters tracking request latency and crash rates.
> * 50 microservices sharing a common service mesh infrastructure.
> * Hundreds of integration test suites exhibiting varying flaky failure rates.
>
> In these multi-group environments, data volume is highly uneven: newly
> deployed clusters have gathered only $n = 2$ runs, while mature anchor
> clusters have processed $n = 400$ requests.
>
> When engineers prompt Large Language Models (LLMs) to analyze multi-group
> telemetry, AI assistants routinely fall into two severe traps:
> 1. **The No-Pooling Small-Sample Trap**: Ranking clusters by raw empirical
>    maximum likelihood estimates (MLE). An uncalibrated AI flags a canary
>    cluster with $1/2$ failures ($50\%$) as the "worst incident in the fleet",
>    triggering unwarranted rollback alerts.
> 2. **Neal's Funnel Sampling Pathology**: Specifying hierarchical models in
>    centered coordinates ($\theta_j \sim \mathcal{N}(\mu, \tau)$). When
>    between-group variance $\tau$ is small, Hamiltonian Monte Carlo (HMC)
>    samplers get trapped in the funnel's narrow neck, producing silent bias
>    and false convergence.
>
> This module provides declarative prompt contracts and verification gates to
> force AI agents to implement partial pooling with non-centered
> parameterization.


## Part 1: The Naive Prompt Trap & Forensic Post-Mortem

### The Prompt That Fools the AI

A DevOps engineer needs to monitor failure rates across 12 regional data center
clusters. Sample sizes are highly imbalanced: Cluster 1 is a newly booted canary
cluster with only $n_1 = 2$ runs ($1$ failure, $50\%$ empirical rate), whereas
Cluster 12 is a mature region with $n_{12} = 400$ runs ($18$ failures,
$4.5\%$ empirical rate).

The engineer submits this prompt to an AI assistant:

```text
We have 12 regional clusters with failure data. Cluster 1 has 1 failure out of
2 requests (50%), while Cluster 12 has 18 failures out of 400 requests (4.5%).
Rank our 12 clusters to identify which ones need emergency remediation. If
possible, fit a hierarchical Bayesian model in PyMC to get true failure rates.
```

---

### The Plausible AI Hallucination

The AI assistant provides a two-part response:

1. **The Flawed Ranking**:
   > *"CRITICAL INCIDENT DETECTED: Cluster 1 is currently your worst-performing
   > region with an alarming 50.0% failure rate! We recommend immediately
   > draining traffic from Cluster 1. Cluster 12 is performing well at 4.5%."*

2. **The Flawed Centered Code**:
```python
# AI Generated Hierarchical Model (Centered Trap!)
import pymc as pm

with pm.Model() as model:
    mu_pop = pm.Normal("mu_pop", mu=0.0, sigma=1.0)
    tau_pop = pm.HalfNormal("tau_pop", sigma=1.0)
    
    # Centered Parameterization: Causes Neal's Funnel!
    theta = pm.Normal("theta", mu=mu_pop, sigma=tau_pop, shape=12)
    p = pm.Deterministic("p", pm.math.invlogit(theta))
    
    y_obs = pm.Binomial("y_obs", n=total_requests, p=p, observed=failed_requests)
    trace = pm.sample(2000, target_accept=0.80)
```

During sampling, the console outputs: `There were 342 divergences after tuning.`
The AI simply advises: *"You can safely ignore these warnings or increase
target_accept to 0.95."*

---

### Forensic Post-Mortem: Why the System Crashes

```
             NEAL'S FUNNEL: THE GEOMETRIC SAMPLING TRAP
 Log(τ)
   2.0 |                 .  *  .  *   .   (Wide exploration)
   1.0 |               .   *   .   *
   0.0 |              .  *   .   *  .
  -1.0 |             .  *   .  *  .
  -2.0 |            .  *  .  * .
  -3.0 |           . *  . * .
  -4.0 |          .*. *.*
  -5.0 |         *!*!*!*!*  <--- Divergences! Leapfrog steps
       +---------------------------- step size ε cannot penetrate
      -3        -1   0    1         3  the infinite curvature neck!
                     θ_j
```

#### 1. The Small-Sample Outlier Illusion (Stein's Paradox)
Raw sample means $\bar{y}_j = k_j / n_j$ have variance inversely proportional to
sample size:
$$\operatorname{Var}(\bar{y}_j) = \frac{p_j (1 - p_j)}{n_j}$$
For Cluster 1 ($n_1 = 2$), the standard error is $\sqrt{0.5 \times 0.5 / 2} =
35.4\%$. The observation $1/2$ is dominated by sampling noise! Under Stein's
Paradox, the sample mean vector is **inadmissible** in $J \ge 3$ dimensions:
partial pooling shrinks extreme outliers toward the grand mean, achieving lower
total expected quadratic error.

In partial pooling, each cluster's estimate is a weighted balance:
$$\hat{\theta}_j = w_j \bar{y}_j + (1 - w_j) \mu_{\text{pop}}, \qquad w_j = \frac{n_j}{n_j + \kappa}$$
Where $\kappa = \sigma^2 / \tau_{\text{pop}}^2$ represents the effective sample
size of the prior. For Cluster 1, $w_1 \approx 0.08$. Its calibrated posterior
failure rate is **$5.3\%$**, completely invalidating the emergency drain alert!

#### 2. The Centered Parameterization Pathology (Neal's Funnel)
In the centered formulation $\theta_j \sim \mathcal{N}(\mu, \tau^2)$, the
prior variance of $\theta_j$ depends dynamically on $\tau$. As $\tau \to 0$, the
prior collapses into a knife-edge spike.

In gradient-based MCMC (HMC/NUTS), the Hamiltonian particle requires an
infinitesimally small step size $\epsilon$ to navigate the neck. Fixed step
sizes overshoot the manifold, generating hundreds of **divergent transitions**.
The sampler fails to explore the low-$\tau$ region, yielding severely biased
shrinkage estimates.

**The Calibrated Fix: Non-Centered Parameterization**
$$\theta_j = \mu_{\text{pop}} + \tau_{\text{pop}} \cdot z_j, \qquad z_j \sim \mathcal{N}(0, 1)$$
By expressing $\theta_j$ as a deterministic transformation of standard normal
auxiliary variables $z_j$, the joint prior geometry becomes a perfectly
spherical, isotropic Gaussian distribution with zero curvature funnels.


## Part 2: Prompting as Probabilistic Programming — Declarative Specifications

To eliminate AI statistical hallucinations, we must specify the generative
physics, prior distributions, and decision loss rather than asking vague
questions. Depending on your workflow, you can specify models using three
progressively simpler patterns:

---

### Level 1: The Everyday 3-Sentence Micro-Spec *(Fastest for Chat)*
Copy and paste these 3 plain sentences into any LLM:

```markdown
1. Data: Failure rates across J = 12 regional clusters with unequal sample
   sizes (n_j in [2, 400]).
2. Architecture: Non-centered hierarchical parameterization theta_j = mu_pop + tau_pop * z_j,
   where z_j ~ Normal(0, 1), to prevent Neal's Funnel divergences.
3. Output: Report shrinkage weights w_j = n_j / (n_j + kappa) and partial-pooling
   estimates to avoid false alarms on small clusters.
```

---

### Level 2: The Model Formula Shorthand *(Concise Probabilistic DSL)*
For quick prompt engineering with statistical models:

```text
Model:     y_j ~ Binomial(n_j, theta_j),  theta_j = mu_pop + tau_pop * z_j
Structure: z_j ~ Normal(0, 1),  tau_pop ~ Half-Normal(0, 1)
Shrinkage: theta_j_shrink = w_j * y_bar_j + (1 - w_j) * mu_pop
```

---

### Level 3: The Socratic Interviewer Meta-Prompt *(Zero Effort)*
If you do not want to write the spec yourself, force the AI to interview you:

> *"Act as a Hierarchical Modeling Architect. I am ranking 12 microservice
clusters with sample sizes ranging from 2 to 400. Ask me 3 questions about
partial pooling, population variance, and Neal's Funnel before ranking them."*

---

<details>
<summary><b>📋 View Full Production YAML Contract (For Automated CI/CD Pipelines & Autonomous Agents)</b></summary>

```yaml
declarative_probabilistic_specification:
  module: "hierarchical_models_and_partial_pooling"
  domain: "multi_group_fleet_reliability"
  
  group_structure:
    entity_name: "regional_cluster"
    group_count: 12
    sample_sizes: "heterogeneous (n_j in [2, 400])"
    response: "failures k_j ~ Binomial(n_j, p_j)"

  hierarchical_architecture:
    parameterization: "non_centered"
    latent_scale: "logit"
    equations:
      auxiliary_standard_normals: "z_j ~ Normal(0, 1), for j = 1..12"
      group_logit_rate: "theta_j = mu_pop + tau_pop * z_j"
      natural_probability: "p_j = inv_logit(theta_j)"

  hyperpriors:
    population_mean_mu: "Normal(mu=-3.0, sigma=0.5)  # Baseline ~ 4.7% failure"
    between_group_spread_tau: "HalfNormal(sigma=0.4) # Moderate cluster variation"

  sampling_invariants:
    sampler: "NUTS / HMC"
    chains: 4
    draws_per_chain: 2000
    target_accept: 0.90
    max_divergences_allowed: 0
    max_rhat: 1.01
    min_ess_bulk: 1000
    min_ess_tail: 1000

  analytical_shrinkage_diagnostics:
    compute_shrinkage_weights: true
    shrinkage_formula: "w_j = n_j / (n_j + kappa)"
    assert_monotonic_shrinkage: "abs(theta_j - y_bar_j) must decrease as n_j grows"

  decision_contract:
    alert_condition: "Alert only if P(p_j > 0.10 | Data) >= 0.95"
```

</details>


## Part 3: The 4 Mandatory Agentic Verification Gates

Before accepting hierarchical model outputs from an AI assistant, execute these
four verification gates:

```
        THE 4 MANDATORY AGENTIC VERIFICATION GATES (CH 13)
 ┌───────────────────────┐      ┌─────────────────────────┐
 │ Gate 1: Non-Centered  │ ───► │ Gate 2: Zero Divergence │
 │ Verify θ = μ + τ * z  │      │ Divergences == 0        │
 │ Auxiliary z ~ N(0, 1) │      │ Target accept >= 0.90   │
 └───────────────────────┘      └─────────────────────────┘
             │                               │
             ▼                               ▼
 ┌───────────────────────┐      ┌─────────────────────────┐
 │ Gate 3: Monotonicity  │ ───► │ Gate 4: Stein's Risk    │
 │ Small-n groups shrink │      │ Total MSE partial-pool  │
 │ strictly more than big│      │ < Total MSE no-pool     │
 └───────────────────────┘      └─────────────────────────┘
```

### Gate 1: Non-Centered Parameterization Architecture Gate
* **Objective**: Confirm that the agent did not write centered code.
* **Criterion**: Scan the model definition. Assert that group parameters
  $\theta_j$ are defined as deterministic affine transforms:
  `theta = mu_pop + tau_pop * z`, where `z` is drawn from a standard normal
  distribution independent of `tau_pop`.

### Gate 2: Zero-Divergence Sampling Invariant Gate
* **Objective**: Verify that the sampler did not get trapped in Neal's Funnel.
* **Criterion**: MCMC sampling across 4 chains must complete with:
  $$\text{Divergent Transitions} = 0$$
  $$\hat{R} \le 1.005 \quad \forall j \in \{1, \dots, J\}, \quad \text{ESS}_{\text{bulk}} > 1,500$$
  If even a single divergence occurs, reject the trace immediately.

### Gate 3: Monotonic Shrinkage Audit Gate
* **Objective**: Ensure shrinkage scales correctly with sample size.
* **Criterion**: Compute empirical shrinkage distance:
  $$\Delta_j = \left|\hat{p}_j^{\text{partial}} - \bar{y}_j^{\text{raw}}\right|$$
  Verify that $\operatorname{Corr}(\Delta_j, n_j) \le -0.80$. Small sample
  clusters must experience substantially larger shrinkage displacements toward
  the population mean than high-volume anchor clusters.

### Gate 4: Stein's Risk Dominance Verification
* **Objective**: Validate empirical error reduction over no-pooling.
* **Criterion**: Under simulation-based calibration with 100 synthetic datasets,
  compute sum of squared parameter errors across all $J = 12$ groups:
  $$\text{SSE}_{\text{partial}} = \sum_{j=1}^{12} (\hat{\theta}_j^{\text{partial}} - \theta_j^{\text{true}})^2, \qquad \text{SSE}_{\text{no}} = \sum_{j=1}^{12} (\bar{y}_j - \theta_j^{\text{true}})^2$$
  $$\mathbb{E}[\text{SSE}_{\text{partial}}] < 0.70 \times \mathbb{E}[\text{SSE}_{\text{no}}]$$


## Part 4: Worked Agent Execution Transcript & Telemetry

### Dataset: 12 Heterogeneous Regional Clusters
* Baseline hyperparameters: $\mu_{\text{pop}} = -3.00$ ($\approx 4.74\%$), 
  $\tau_{\text{pop}} = 0.35$.
* True fleet failure rate $\approx 4.5\text{--}5.0\%$.

```text
================================================================================
           12-CLUSTER HIERARCHICAL PARTIAL POOLING TELEMETRY LOG
================================================================================
Cluster ID   Runs (n)   Fails (k)   Raw Rate   Shrink Wt (w)  Partial Mean   95% Credible Interval
-------------------------------------------------------------------------------------------------
Cluster 01          2           1    50.00%           0.082         5.38%        [2.11%, 10.42%]
Cluster 02          5           1    20.00%           0.183         5.21%        [2.24%,  9.68%]
Cluster 03         10           0     0.00%           0.310         3.72%        [1.65%,  6.82%]
Cluster 04         15           1     6.67%           0.402         4.94%        [2.38%,  8.41%]
Cluster 05         25           2     8.00%           0.528         5.48%        [2.89%,  9.12%]
Cluster 06         40           1     2.50%           0.642         3.68%        [1.85%,  6.15%]
Cluster 07         60           3     5.00%           0.730         4.86%        [2.75%,  7.58%]
Cluster 08         90           4     4.44%           0.801         4.52%        [2.68%,  6.89%]
Cluster 09        140           8     5.71%           0.862         5.49%        [3.52%,  7.98%]
Cluster 10        200          11     5.50%           0.899         5.39%        [3.68%,  7.51%]
Cluster 11        300          12     4.00%           0.931         4.12%        [2.85%,  5.69%]
Cluster 12        400          18     4.50%           0.947         4.52%        [3.28%,  6.02%]
================================================================================
Sampler: NUTS (Non-Centered) | Chains: 4 | Draws: 8000 | Divergences: 0
Max R-hat: 1.0004 | Min ESS Bulk: 6,412 | Min ESS Tail: 5,910
================================================================================
```

---

### The Anatomy of Shrinkage: Visual Comparison

```
              SHRINKAGE DYNAMICS ACROSS 12 CLUSTERS
 Failure
   Rate
   50% |   * [Cluster 1: Raw n=2 (1/2)]
       |    \
   20% |     \      * [Cluster 2: Raw n=5 (1/5)]
       |      \    /
       |       \  /
    8% |        \/        * [Cluster 5: n=25]
    5% |=========●========●=======●======●=======●====== [Population Mean: 4.8%]
    4% |                                  \     /
    2% |                                   *   * [Cluster 6: n=40]
    0% |           * [Cluster 3: n=10 (0/10)]
       +---------------------------------------------------
           Small Sample (n < 20)           Large Sample (n > 100)
             HEAVY SHRINKAGE                   MINIMAL SHRINKAGE
```

### Decision Alert Telemetry
* Operational Policy: Alert if $P(p_j > 10\% \mid \text{Data}) \ge 95\%$.
* Under Naive AI (Raw MLE): Triggered emergency paging on Cluster 1 ($50\%$)
  and Cluster 2 ($20\%$).
* Under Calibrated Hierarchical Agent:
  * Cluster 1: $P(p_1 > 10\%) = 2.8\%$ (NO ALERT).
  * Cluster 2: $P(p_2 > 10\%) = 1.9\%$ (NO ALERT).
  * Zero clusters breached the threshold. Fleet operations saved from false
    emergency drain.


## Part 5: Naive AI Prompt vs. Calibrated Epistemic Specification

| Architectural Aspect | Naive AI Assistant Output | Calibrated Probabilistic Specification |
| :--- | :--- | :--- |
| **Multi-Group Estimation** | Independent No-Pooling MLE $\bar{y}_j = k_j/n_j$ | Bayesian Partial Pooling via adaptive shrinkage weights |
| **Small-Sample Vulnerability**| Flags $n=2$ canary cluster ($50\%$) as worst crisis | Correctly shrinks $n=2$ cluster to $5.38\%$ ($w_1 = 0.082$) |
| **Model Parameterization** | Centered: $\theta \sim \mathcal{N}(\mu, \tau)$ | Non-centered: $\theta = \mu + \tau \cdot z$, $z \sim \mathcal{N}(0, 1)$ |
| **HMC Geometry** | Trapped in Neal's Funnel; hundreds of divergences | Isotropic spherical space; zero divergences guaranteed |
| **Between-Group Spread** | Hardcoded or ignored | Learned adaptively from collective multi-group data |
| **Total Parameter Error** | High total MSE (inadmissible in $J \ge 3$) | Dominates No-Pooling with $>30\%$ error reduction |
| **Decision Alert Policy** | Paged on raw ratio $k/n > 0.10$ | Alerts on tail integral $P(p_j > 0.10 \mid \text{Data}) \ge 0.95$ |


## Part 6: 🎯 Check Your Understanding — 3-Tier "Audit the AI" Exercises

Test your skills in auditing AI-generated hierarchical probabilistic models.

---

### 🟢 Tier 1: Spot the AI Hallucination
An AI agent inspects automated test telemetry across 20 software test suites.
Suite A had 0 failures in 3 runs ($0\%$). Suite B had 50 failures in 1,000 runs
($5\%$).

The AI reports:
> *"Test Suite A is completely stable (0.0% failure rate) and requires zero
> attention. Test Suite B is unstable and should be quarantined."*

**Task**: Identify the epistemic hallucination in the agent's logic.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:
The agent fell into the small-sample zero-count fallacy. An empirical record of
$0/3$ does not prove a failure rate of $0\%$. The $95\%$ credible interval for
Suite A under a weakly informative prior $\text{Beta}(1, 1)$ or hierarchical
shrinkage is $[0.01, 0.45]$—meaning its true failure rate could easily be $15\%$
or higher!

Conversely, Suite B has an extremely precise estimate of $5.0\% \pm 0.7\%$.
Labeling Suite A as "completely stable" while ignoring its massive epistemic
uncertainty is a classic AI failure mode that partial pooling resolves.
</details>

---

### 🟡 Tier 2: Refactor the Flawed Specification
An engineer drafted this PyMC prompt:

```text
Build a hierarchical model for customer conversion across 50 marketing campaigns:
mu ~ Normal(0, 1)
tau ~ Exponential(1)
theta_j ~ Normal(mu, tau)  # for j in 1..50
conv_j ~ Binomial(n_j, logit^{-1}(theta_j))
```

**Task**: Rewrite this model using non-centered parameterization and write the
verification assertion that rejects traces with divergences.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:

```python
import pymc as pm

with pm.Model() as non_centered_model:
    # Hyperpriors
    mu_pop = pm.Normal("mu_pop", mu=0.0, sigma=1.0)
    tau_pop = pm.Exponential("tau_pop", lam=1.0)
    
    # Non-Centered Auxiliary Standard Normals
    z = pm.Normal("z", mu=0.0, sigma=1.0, shape=50)
    
    # Deterministic Affine Reconstruction (Escapes Neal's Funnel!)
    theta = pm.Deterministic("theta", mu_pop + tau_pop * z)
    p = pm.Deterministic("p", pm.math.invlogit(theta))
    
    # Likelihood
    conv_obs = pm.Binomial("conv_obs", n=n_campaigns, p=p, observed=k_conversions)
    
    # Sampler configuration with verification gate
    trace = pm.sample(2000, target_accept=0.90)

# Mandatory Verification Gate Assertion
divergences = trace.sample_stats["diverging"].values.sum()
assert divergences == 0, f"FAIL: Sampler suffered {divergences} divergences!"
print("Verification Passed: Clean non-centered sampling with zero divergences.")
```
</details>

---

### 🔴 Tier 3: Architect an Autonomous Agent Contract
You are designing an autonomous site reliability agent that monitors 100 backend
microservices. The agent must automatically compute shrinkage weights $w_j$ for
each microservice, rank them by posterior probability of degradation, and
guarantee zero divergences.

**Task**: Write the declarative contract specifying the non-centered model,
shrinkage telemetry reporting, and decision rule.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:

```yaml
autonomous_agent_contract:
  role: "Fleet Service Reliability Shrinkage Agent"
  objective: "Calibrated microservice ranking without small-sample false alarms"
  
  model_specification:
    group_count: 100
    parameterization: "non_centered"
    priors:
      mu_pop: "Normal(mu=-3.5, sigma=0.5)"
      tau_pop: "HalfNormal(sigma=0.3)"
    equations:
      - "z_j ~ Normal(0, 1), j = 1..100"
      - "theta_j = mu_pop + tau_pop * z_j"
      - "latency_rate_j = inv_logit(theta_j)"
      
  diagnostic_invariants:
    zero_divergence_rule: "Divergent transitions must equal 0"
    max_rhat: 1.005
    min_ess_bulk: 1500
    shrinkage_audit:
      formula: "w_j = n_j / (n_j + (sigma^2 / tau_pop^2))"
      report: "Output table of (service_id, n_j, raw_rate, w_j, posterior_rate)"
      
  decision_rule:
    quarantine_criterion: "P(latency_rate_j > 0.08 | Data) >= 0.95"
    action: "Mark microservice for traffic throttling if quarantine_criterion is true"
```
</details>


## Part 7: Key Takeaways & Navigation

### Core Architectural Rules for Multi-Group AI Systems
1. **Never Rank Entities by Raw Sample Ratios**: Raw MLEs place the highest
   variance (smallest sample) groups at the extremes of the leaderboard. Always
   use partial pooling to shrink noisy estimates toward the population mean.
2. **Always Enforce Non-Centered Parameterization**: In HMC, centered
   formulations $\theta_j \sim \mathcal{N}(\mu, \tau)$ get trapped in Neal's
   Funnel whenever $\tau$ is small. Express parameters as $\theta_j = \mu + \tau
   \cdot z_j$ with $z_j \sim \mathcal{N}(0, 1)$.
3. **Audit Divergences as Hard Blockers**: A non-zero divergence count is not a
   harmless warning; it is mathematical proof that the MCMC sampler cannot
   explore high-curvature posterior regions.
4. **Leverage Stein's Paradox**: Shrinking group means toward a common anchor
   guarantees lower total expected error than treating groups as isolated
   silos.

---

### 🧭 Course Navigation

* 🏠 **[Course Home & Master Syllabus](../README.md)**
* ⬅️ **Previous**: **[Ch 12: Bayesian Regression, Counts & GLMs](12_bayesian_regression_and_glms.ipynb)** *(Part III: Applied Modeling, Regression, Hierarchy & Model Comparison)*
* ➡️ **Next**: **[Ch 14: Marginal Likelihood & Occam's Razor](14_marginal_likelihood_and_model_comparison.ipynb)** *(Part III: Applied Modeling, Regression, Hierarchy & Model Comparison)*
* 🔄 **Companion Tracks for Chapter 13**: [📖 Conceptual](../conceptual/13_hierarchical_models_and_partial_pooling.ipynb) | [🤖 AI Track](13_hierarchical_models_and_partial_pooling.ipynb) | [🐍 Python](../python/13_hierarchical_models_and_partial_pooling.ipynb) | [🔵 R](../r/13_hierarchical_models_and_partial_pooling.ipynb)
